# Level 2 프로젝트 스타터: Autonomous Vision Agent

과제:

> Find and sort the six cubes in the warehouse into their matching destination pads.

Level 2에서는 scene_state, 정확한 entity ID, coordinate go_to를 사용할 수 없습니다. camera observation, set_head, set_velocity, memory, recovery로 이동합니다.

실행 전에 `MENLO_API_KEY`와 `TOKAMAK_API_KEY`를 모두 설정하세요. 프로젝트는 text LLM decision loop를 필수로 요구합니다.


## 사용 방법

아래 코드는 완성된 solution이 아니라 최소 scaffold입니다. SUPPORT CODE는 setup, wrapper, schema, memory 구조를 제공합니다. STUDENT TODO 영역에서 LLM decision, observation, action execution, verification, memory update 전략을 직접 설계하세요.


In [ ]:
# Colab/로컬 setup. 필요하면 한 번 실행하세요.
%pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib


## 스타터 Scaffold

이 cell을 실행한 뒤 TODO 영역을 팀 설계에 맞게 수정하세요.


In [ ]:
"""Level 2 프로젝트 스타터입니다.

이 파일은 완성된 해답이 아니라 최소 scaffold입니다.

SUPPORT CODE 영역은 반복해서 작성할 필요가 없는 wrapper, 자료 구조,
schema validation을 제공합니다. STUDENT TODO 영역은 팀이 직접 설계하고,
개선하고, 테스트하고, 발표에서 설명해야 하는 부분입니다.

Level 2 규칙: `scene_state`, 정확한 entity ID, coordinate `go_to`를 사용할 수
없습니다. 카메라 관찰값, `set_head`, `set_velocity`, memory, recovery로
navigation을 구현해야 합니다.
"""

import asyncio
import json
import math
from dataclasses import dataclass, field
from typing import Any

from menlo_runner.llm import ask_vlm
from menlo_runner.perception import detect_color_blobs


# ---------------------------------------------------------------------------
# SUPPORT CODE: 공통 과제 정의와 필수 LLM decision schema
# ---------------------------------------------------------------------------
TASK = "Find and sort the six cubes in the warehouse into their matching destination pads."

DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
SIGNAGE_NOTE = (
    "A is the conveyor/cube source area, not a destination. "
    "Destination signs are B red, C green, D blue, E yellow."
)

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    """LLM이 반환하고 코드가 검증한 고수준 decision입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 유지하는 agent 상태입니다."""

    delivered_count: int = 0
    delivery_limit: int | None = None
    priority_colors: list[str] = field(default_factory=list)
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """LLM과 action code에 전달할 compact observation입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """head pose를 함께 저장한 색상 detection입니다."""

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float

    @property
    def full_bearing_deg(self) -> float:
        """image angle에 head yaw를 더한 대략적인 body-relative bearing입니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """LLM의 JSON 응답을 parse하고 필수 schema를 검증합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """robot state를 LLM에 전달하기 좋은 compact text context로 변환합니다."""
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "active_color": memory.active_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "delivery_limit": memory.delivery_limit,
        "priority_colors": memory.priority_colors,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "vlm_summary": observation.vlm_summary,
    }


# ---------------------------------------------------------------------------
# SUPPORT CODE: Level 2에서 허용되는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 wrapper에는 scene_state, 정답 좌표, 정확한 cube/pad entity ID, go_to를
# 추가하지 마세요.

async def get_robot_status(ctx: Any) -> Any:
    """robot pose, motion status, neck state를 읽습니다."""
    return await ctx.state("robot_status")


async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다."""
    return await ctx.get_vision("pov")


def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 창고 표지판을 읽기 위한 VLM prompt입니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" The robot is holding a {held_color} cube, so the target destination sign is {DESTINATION_SIGN_RULES[held_color]}."
    return (
        "Read the floating warehouse signs visible in this robot camera frame. "
        f"{SIGNAGE_NOTE} "
        "Return JSON with visible sign letters, colors, rough left/center/right positions, and confidence."
        + target
    )


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """현재 POV frame에 대해 project-allowed VLM helper로 질문합니다."""
    jpeg = await get_camera_frame(ctx)
    return ask_vlm(jpeg, prompt, api_key=api_key)


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    jpeg = await get_camera_frame(ctx)
    return detect_color_blobs(jpeg)


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """walking direction은 바꾸지 않고 camera 방향만 조정합니다."""
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await ctx.invoke("set_head", args, timeout_s=10)


async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command를 보냅니다."""
    return await ctx.invoke(
        "set_velocity",
        {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
        timeout_s=30,
    )


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {})


async def pick_nearest_cube(ctx: Any) -> Any:
    """의도한 cube 가까이 시각적으로 이동한 뒤 nearest cube를 집습니다."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone(ctx: Any) -> Any:
    """matching pad 가까이 이동한 뒤 nearest zone에 내려놓습니다."""
    return await ctx.invoke("place_entity", {}, timeout_s=300)


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log에 넣기 쉬운 작은 dictionary로 변환합니다."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. 더 나은 search 전략으로 교체할 수 있습니다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        await set_head(ctx, yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        for detection in await perceive(ctx):
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                )
            )
    return all_detections


# ---------------------------------------------------------------------------
# STUDENT TODO: LLM decision 함수
# ---------------------------------------------------------------------------

async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """LLM으로 다음 고수준 action을 선택하고, 실패 시 안전 fallback을 사용합니다."""
    decision_context = build_decision_context(task, observation, memory, last_result)
    system_prompt = "\n".join(
        [
            "You are the high-level planner for a Level 2 vision-only warehouse robot.",
            "Return ONLY one JSON object. Do not include markdown.",
            "Allowed next_action values: " + ", ".join(sorted(ALLOWED_NEXT_ACTIONS)) + ".",
            'Schema: {"next_action": "...", "target_color": null, "reason": "...", "recovery_strategy": null}',
            "Do not output velocity values, coordinates, entity ids, or low-level robot commands.",
            "Use the current stage/memory: need_cube -> approach/pick -> holding_cube -> pad/place -> stop.",
        ]
    )

    parsed: AgentDecision | None = None
    try:
        from menlo_runner.llm import call_llm  # type: ignore

        reply = call_llm(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(decision_context, ensure_ascii=False)},
            ]
        )
        parsed = parse_agent_decision(reply)
    except Exception as exc:
        memory.logs.append({"llm_fallback": type(exc).__name__, "stage": memory.stage})

    if parsed is not None:
        return parsed

    visible = decision_context["visible_targets"]
    target_color = memory.held_color or memory.active_color
    matching_visible = [item for item in visible if target_color is None or item["color"] == target_color]
    best = max(matching_visible or visible, key=lambda item: item["blob_area"], default=None)
    repeated_failures = max(memory.failed_attempts.values(), default=0)

    if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
        return AgentDecision(next_action="stop", reason="Delivery limit reached.")
    if repeated_failures >= 3:
        return AgentDecision(next_action="recover", reason="Repeated failures require recovery.", recovery_strategy="backoff_rescan")
    if memory.held_color:
        if memory.stage in {"holding_cube", "search_pad"}:
            return AgentDecision(next_action="search_pad", target_color=memory.held_color, reason="Holding cube; find matching pad/sign.")
        if memory.stage == "approaching_pad" and best:
            return AgentDecision(next_action="navigate_to_pad", target_color=memory.held_color, reason="Matching pad color/sign evidence is visible.")
        if memory.stage == "ready_to_place":
            return AgentDecision(next_action="place_cube", target_color=memory.held_color, reason="Near matching pad; place nearest zone.")
        return AgentDecision(next_action="search_pad", target_color=memory.held_color, reason="Fallback while holding cube.")
    if memory.stage == "ready_to_pick" and memory.active_color:
        return AgentDecision(next_action="pick_cube", target_color=memory.active_color, reason="Near selected cube; pick nearest cube.")
    if best:
        return AgentDecision(next_action="navigate_to_cube", target_color=best["color"], reason="Visible cube color selected by camera blob evidence.")
    return AgentDecision(next_action="search_cube", reason="No useful color blob visible; scan for cube.")


# ---------------------------------------------------------------------------
# STUDENT TODO: observation, verification, memory
# ---------------------------------------------------------------------------

async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """현재 stage에 맞춰 camera/head 기반 observation을 수집합니다."""
    robot_status = await get_robot_status(ctx)
    if memory.stage in {"need_cube", "search_cube", "holding_cube", "search_pad", "recover"}:
        detections = await scan_head(ctx)
        note = "wide_scan"
    else:
        await set_head(ctx, yaw=0.0, pitch=0.15)
        await asyncio.sleep(0.2)
        detections = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.15,
            )
            for d in await perceive(ctx)
        ]
        note = "front_frame"
    return Observation(robot_status=robot_status, detections=detections, note=note)


def _result_status(action_result: dict[str, Any]) -> str | None:
    result = action_result.get("result") if isinstance(action_result, dict) else None
    if isinstance(result, dict):
        return result.get("status")
    return action_result.get("status") if isinstance(action_result, dict) else None


def _is_done_status(status: str | None) -> bool:
    return str(status).lower() in {"done", "success", "succeeded", "completed", "ok"}


async def verify_outcome(ctx: Any, decision: AgentDecision, action_result: dict[str, Any]) -> dict[str, Any]:
    """SDK 결과와 직후 camera evidence를 함께 묶어 다음 cycle 판단에 제공합니다."""
    status = _result_status(action_result)
    post_status = await get_robot_status(ctx)
    detections: list[Any] = []
    target_seen = False
    max_target_area = 0

    if decision.next_action not in {"stop", "skip_target"}:
        try:
            detections = await scan_head(ctx, yaws=(-0.35, 0.0, 0.35), pitch=0.15)
            target_seen = any(d.color == decision.target_color for d in detections) if decision.target_color else bool(detections)
            max_target_area = max((d.blob_area for d in detections if decision.target_color is None or d.color == decision.target_color), default=0)
        except Exception as exc:
            return {
                "ok": False,
                "decision": decision.__dict__,
                "action_result": action_result,
                "error": f"post_action_observation_failed:{type(exc).__name__}",
                "sdk_status": status,
            }

    ok = True
    if decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        ok = bool(action_result.get("reached"))
    elif decision.next_action in {"search_cube", "search_pad"}:
        ok = bool(action_result.get("found"))
    elif decision.next_action in {"pick_cube", "place_cube"}:
        ok = _is_done_status(status)
    elif decision.next_action == "recover":
        ok = action_result.get("status") in {"recovered", "stepped_back_and_rotated"}

    return {
        "ok": ok,
        "decision": decision.__dict__,
        "action_result": action_result,
        "sdk_status": status,
        "post_visible_count": len(detections),
        "target_seen": target_seen,
        "max_target_area": max_target_area,
        "robot_status_type": type(post_status).__name__,
    }


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """각 cycle 뒤 stage, active/held color, 실패 횟수, 완료 상태를 갱신합니다."""
    ok = bool(verified.get("ok"))
    color = decision.target_color or memory.active_color or memory.held_color
    failure_key = f"{decision.next_action}:{color or 'none'}"

    if ok:
        memory.failed_attempts.pop(failure_key, None)
    elif decision.next_action != "stop":
        memory.failed_attempts[failure_key] = memory.failed_attempts.get(failure_key, 0) + 1

    if decision.next_action == "search_cube":
        memory.stage = "need_cube" if not ok else "approaching_cube"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_cube":
        memory.active_color = decision.target_color or memory.active_color
        memory.stage = "ready_to_pick" if ok else "search_cube"
    elif decision.next_action == "pick_cube":
        if ok and color:
            memory.held_color = color
            memory.active_color = color
            memory.stage = "holding_cube"
        else:
            memory.stage = "search_cube"
    elif decision.next_action == "search_pad":
        memory.stage = "approaching_pad" if ok else "search_pad"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_pad":
        memory.stage = "ready_to_place" if ok else "search_pad"
    elif decision.next_action == "place_cube":
        if ok:
            delivered_color = memory.held_color or color
            memory.delivered_count += 1
            if delivered_color and delivered_color not in memory.completed_colors:
                memory.completed_colors.append(delivered_color)
            memory.held_color = None
            memory.active_color = None
            memory.stage = "done" if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit else "need_cube"
        else:
            memory.stage = "search_pad"
    elif decision.next_action == "recover":
        memory.stage = "search_pad" if memory.held_color else "search_cube"
    elif decision.next_action == "skip_target":
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.active_color = None
        memory.stage = "holding_cube" if memory.held_color else "need_cube"
    elif decision.next_action == "stop":
        memory.stage = "done"

    memory.logs.append(
        {
            "stage": memory.stage,
            "active_color": memory.active_color,
            "held_color": memory.held_color,
            "delivered_count": memory.delivered_count,
            "observation": {
                "visible_count": len(observation.detections),
                "note": observation.note,
                "colors": sorted({getattr(d, "color", "unknown") for d in observation.detections}),
            },
            "llm_decision": decision.__dict__,
            "verified": verified,
            "failed_attempts": dict(memory.failed_attempts),
        }
    )


# ---------------------------------------------------------------------------
# LEVEL 2 STUDENT TODO: vision-only action 구현
# ---------------------------------------------------------------------------
# Level 2에서는 go_to를 호출하면 안 됩니다. camera observation, set_head,
# set_velocity, memory, recovery behavior로 navigation을 구현하세요.

def _best_detection(detections: list[Any], target_color: str | None = None) -> Any | None:
    candidates = [d for d in detections if target_color is None or d.color == target_color]
    return max(candidates, key=lambda d: d.blob_area, default=None)


async def visual_search(ctx: Any, target_color: str | None = None) -> bool:
    """head scan과 짧은 body rotation으로 target color blob을 찾습니다."""
    for yaws in [(-0.9, -0.45, 0.0, 0.45, 0.9), (-0.6, 0.0, 0.6)]:
        detections = await scan_head(ctx, yaws=yaws, pitch=0.15)
        if _best_detection(detections, target_color) is not None:
            return True
        await move_velocity(ctx, vx=0.12, wz=0.35, duration_s=0.8)
    detections = await scan_head(ctx, yaws=(-0.8, 0.0, 0.8), pitch=0.15)
    return _best_detection(detections, target_color) is not None


async def visual_navigate_to_target(ctx: Any, target_color: str | None) -> bool:
    """색상 blob bearing/area feedback으로 짧은 이동을 반복합니다."""
    if target_color is None:
        return False

    arrival_area = 9000
    centered_tolerance_deg = 9.0
    for _ in range(14):
        detections = await scan_head(ctx, yaws=(-0.35, 0.0, 0.35), pitch=0.15)
        target = _best_detection(detections, target_color)
        if target is None:
            await move_velocity(ctx, vx=0.10, wz=0.30, duration_s=0.6)
            continue

        bearing = getattr(target, "full_bearing_deg", target.angle_deg)
        if target.blob_area >= arrival_area and abs(bearing) <= 18.0:
            await set_head(ctx, yaw=0.0, pitch=0.15)
            return True

        if abs(bearing) > centered_tolerance_deg:
            wz = max(-0.35, min(0.35, -bearing / 90.0))
            await move_velocity(ctx, vx=0.12, wz=wz, duration_s=0.55)
        else:
            speed = 0.28 if target.blob_area < arrival_area * 0.55 else 0.16
            await move_velocity(ctx, vx=speed, wz=0.0, duration_s=0.75)

    return False


async def recover_motion(ctx: Any, memory: AgentMemory, reason: str | None = None) -> dict[str, Any]:
    """반복 실패 횟수에 따라 후진, 회전, 재스캔 폭을 조절합니다."""
    worst_failure = max(memory.failed_attempts.values(), default=0)
    await cancel_action(ctx)
    await move_velocity(ctx, vx=-0.18, duration_s=0.7)
    turn = 0.30 if worst_failure % 2 == 0 else -0.30
    await move_velocity(ctx, vx=0.10, wz=turn, duration_s=1.0)
    detections = await scan_head(ctx, yaws=(-0.9, 0.0, 0.9), pitch=0.15)
    status = "recovered" if detections else "recovered_no_target_seen"
    return {
        "action": "recover",
        "reason": reason,
        "status": status,
        "visible_count": len(detections),
        "worst_failure": worst_failure,
    }


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 LLM decision 하나를 Level 2 robot action으로 변환합니다.

    TODO:
    - go_to 없이 search/navigation을 구현하세요.
    - 의도한 cube 가까이 시각적으로 이동한 뒤 pick하세요.
    - matching pad 가까이 시각적으로 이동한 뒤 place하세요.
    - target을 잃거나 이동에 실패하면 recovery를 사용하세요.
    """
    if decision.next_action in {"search_cube", "search_pad"}:
        found = await visual_search(ctx, decision.target_color)
        return {"action": decision.next_action, "found": found}

    if decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        reached = await visual_navigate_to_target(ctx, decision.target_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "pick_cube":
        result = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "result": result_summary(result)}

    if decision.next_action == "place_cube":
        result = await place_nearest_zone(ctx)
        return {"action": "place_cube", "result": result_summary(result)}

    if decision.next_action == "recover":
        return await recover_motion(ctx, memory, decision.recovery_strategy)

    return {"action": decision.next_action, "status": "no_op"}


async def run_agent(ctx: Any, *, max_cycles: int = 40) -> AgentMemory:
    """observe-LLM-act-verify loop with delivery/retry stop conditions."""
    memory = AgentMemory(delivery_limit=6)
    last_result: dict[str, Any] | None = None

    for cycle in range(1, max_cycles + 1):
        print(f"\n[Level 2] Cycle {cycle}")
        if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
            memory.stage = "done"
            break
        if max(memory.failed_attempts.values(), default=0) >= 5:
            memory.logs.append({"stop_reason": "too_many_repeated_failures", "failed_attempts": dict(memory.failed_attempts)})
            break

        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            update_memory(memory, observation, decision, {"ok": True, "stop_reason": decision.reason})
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        verified = await verify_outcome(ctx, decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified

    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Running Level 2 autonomous-vision project starter")
    memory = await run_agent(ctx)
    print("\nRun complete.")
    print(f"Delivered count: {memory.delivered_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)


## 로봇 연결

Prompt가 나오면 viewer URL을 Chrome에서 여세요.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-2-notebook-ko")
print(ctx.viewer_url)


## Agent 실행

현재 실험에 필요한 TODO를 충분히 구현한 뒤 실행하세요.


In [ ]:
memory = await run_agent(ctx)
memory.logs


In [ ]:
# 종료할 때 cleanup하세요.
# await ctx.close()
